# Web search and multi-hop retrieval (Chapter 8)

This notebook wraps Tavily as a synchronous ReAct tool, combines live and private retrieval, and implements both multi-hop patterns from the chapter: agent-controlled ReAct and a fixed-depth `MultiHop` module.

**Environment variables**

- `OPENAI_API_KEY`
- `TAVILY_API_KEY`


In [ ]:
%pip install -r ../requirements.txt -q


In [ ]:
import dspy
from dotenv import load_dotenv

load_dotenv()

dspy.configure(lm=dspy.LM("openai/gpt-5.6-luna"))


In [ ]:
import os
import requests


TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
if not TAVILY_API_KEY:
    print("Set TAVILY_API_KEY in .env before running live web-search calls.")


## A synchronous Tavily tool


In [ ]:
def web_search(query: str) -> str:
    """Search the web for current information about a topic."""
    if not TAVILY_API_KEY:
        return "web_search unavailable: set TAVILY_API_KEY in your environment."

    response = requests.post(
        "https://api.tavily.com/search",
        headers={"Authorization": f"Bearer {TAVILY_API_KEY}"},
        json={"query": query, "max_results": 3},
        timeout=30,
    )
    response.raise_for_status()

    results = response.json().get("results", [])
    return "\n\n".join(
        f"{result['title']}\n"
        f"{result['url']}\n"
        f"{result['content']}"
        for result in results
    )


def search_knowledge_base(query: str) -> str:
    """Search our internal product documentation."""
    return f"[stub] Our refund policy allows returns within 30 days. Query: {query}"


def calculate(expression: str) -> str:
    """Evaluate a mathematical expression and return the result."""
    return str(eval(expression))


## Mix private retrieval, web search, and calculation


In [ ]:
agent = dspy.ReAct(
    "question -> answer, sources: list[str]",
    tools=[
        search_knowledge_base,
        web_search,
        calculate,
    ],
    max_iters=10,
)

result = agent(
    question="Compare our refund policy with current FTC guidance."
)
print(result.answer)
print(result.sources)


## Approach 1: ReAct with retrieval tools

ReAct controls whether to search again and how to rewrite each query. Cost and latency vary with the number of iterations.


In [ ]:
multihop_agent = dspy.ReAct(
    "question -> answer, sources: list[str]",
    tools=[
        search_knowledge_base,
        web_search,
    ],
    max_iters=12,
)


## Approach 2: fixed-depth `MultiHop`

For a predictable retrieval budget, generate a query, integrate its passages into deduplicated notes, and repeat exactly `num_hops` times. This demo uses the same in-memory retriever interface introduced earlier in the chapter.


In [ ]:
corpus = [
    "DSPy grew from research on compiling declarative language-model programs.",
    "DSPy signatures separate task intent from prompts and model-specific formatting.",
    "Later DSPy releases added ReAct, typed tools, optimizers, and trajectory evaluation.",
    "Agent systems benefit when tool-use trajectories can be measured and optimized.",
]

embedder = dspy.Embedder(
    "openai/text-embedding-3-small",
    dimensions=512,
)
search = dspy.retrievers.Embeddings(
    embedder=embedder,
    corpus=corpus,
    k=3,
)


class MultiHop(dspy.Module):
    def __init__(self, retriever, num_hops=3):
        super().__init__()
        self.retriever = retriever
        self.num_hops = num_hops

        self.generate_query = dspy.ChainOfThought(
            "question, notes -> query"
        )
        self.integrate = dspy.ChainOfThought(
            "question, notes, context -> new_notes: list[str]"
        )
        self.respond = dspy.ChainOfThought(
            "question, notes -> answer"
        )

    def forward(self, question):
        notes, sources = [], []

        for _ in range(self.num_hops):
            query = self.generate_query(
                question=question,
                notes=notes,
            ).query

            context = self.retriever(query).passages
            pred = self.integrate(
                question=question,
                notes=notes,
                context=context,
            )

            notes.extend(
                note
                for note in pred.new_notes
                if note not in notes
            )
            sources.extend(
                passage
                for passage in context
                if passage not in sources
            )

        answer = self.respond(
            question=question,
            notes=notes,
        ).answer

        return dspy.Prediction(
            answer=answer,
            notes=notes,
            sources=sources,
        )


multi_hop = MultiHop(
    retriever=search,
    num_hops=3,
)
multi_hop_result = multi_hop(
    question="How did DSPy influence later agent systems?"
)
print(multi_hop_result.answer)
print(multi_hop_result.sources)
